# Reproduce SSO Errors (Client: Livia Hull, Jan 2026)

This notebook reproduces the three SSO errors from the client email thread,
then tests the fixed flow.

| Error | Description | Root Cause |
|-------|-------------|------------|
| P0 | "Code verifier required" | Django LocMemCache loses PKCE verifier across workers |
| P1 | "State is invalid" | Blocking `sso_timeout=50` polls before user completes login |
| P2 | `login()` missing username/password | Client called wrong API method |

## Instructions

1. Run **Cell 1** to install dependencies
2. Set your server in **Cell 2**
3. Run each error cell to reproduce
4. Run **Cell 7** to test the fixed flow

In [ ]:
# Cell 1 — Install dependencies
%pip install graphistry requests
dbutils.library.restartPython()

In [ ]:
# Cell 2 — Configuration

SERVER = "localhost"       # Change to your Graphistry server
PROTOCOL = "http"          # "https" for remote servers
ORG_NAME = None            # Organization name, or None for site-wide
IDP_NAME = None            # IdP name, or None for default

BASE_URL = f"{PROTOCOL}://{SERVER}"
print(f"Target: {BASE_URL}")

## Error 1 (P0): "Code verifier required"

The server stores a PKCE code verifier in Django's cache as `sso_cv_{state}` during
the authorize step. With `LocMemCache` (per-process), if a different Gunicorn worker
handles the Okta callback, the verifier is gone. Okta rejects the token exchange.

**To reproduce:** Run Cell 3, click the SSO link, complete login.
- With **LocMemCache**: The callback page shows "Code verifier required"
- With **Redis** (fix applied): Login succeeds

In [ ]:
# Cell 3 — Error 1: "Code verifier required"
#
# Initiates SSO, then you click the link and complete login.
# If LocMemCache is the default cache, the callback fails.

import requests as req
import json
from urllib.parse import parse_qs, urlparse
from IPython.display import display, HTML

def sso_login_url(base, org=None, idp=None):
    if org is None and idp is None:
        return f"{base}/api/v2/g/sso/oidc/login/"
    elif org is not None and idp is None:
        return f"{base}/api/v2/o/{org}/sso/oidc/login/"
    elif org is not None and idp is not None:
        return f"{base}/api/v2/o/{org}/sso/oidc/login/{idp}/"
    return f"{base}/api/v2/g/sso/oidc/login/"

def token_poll_url(base, state):
    return f"{base}/api/v2/o/sso/oidc/jwt/{state}/"

# Initiate SSO
url = sso_login_url(BASE_URL, ORG_NAME, IDP_NAME)
resp = req.post(url, timeout=15)
body = resp.json()
data = body.get("data", {})
error1_state = data.get("state", "")
error1_auth_url = data.get("auth_url", "")

if error1_state and error1_auth_url:
    params = parse_qs(urlparse(error1_auth_url).query)
    challenge = params.get("code_challenge", [None])[0]
    method = params.get("code_challenge_method", [None])[0]

    display(HTML(f'<h3>Error 1: Code verifier required</h3>'))
    display(HTML(f'<p>State: <code>{error1_state}</code></p>'))
    display(HTML(f'<p>PKCE: <code>code_challenge_method={method}</code></p>'))
    display(HTML(f'<p>Server cached <code>sso_cv_{error1_state[:8]}...</code> in Django default cache</p>'))
    display(HTML(f'<hr/>'))
    display(HTML(f'<p><strong>Click this link to complete SSO login:</strong></p>'))
    display(HTML(f'<a href="{error1_auth_url}" target="_blank">Login via SSO</a>'))
    display(HTML(f'<hr/>'))
    display(HTML(f'<p>Expected with <strong>LocMemCache</strong>: Error page &rarr; "Code verifier required"</p>'))
    display(HTML(f'<p>Expected with <strong>Redis</strong> (fix applied): Successful redirect to Graphistry</p>'))
else:
    error_msg = body.get("message") or body.get("error") or json.dumps(body)
    display(HTML(f'<p>&#10060; SSO initiation failed: {error_msg}</p>'))

In [ ]:
# Cell 4 — Check Error 1 result (run AFTER completing SSO login above)

poll_url = token_poll_url(BASE_URL, error1_state)
resp = req.get(poll_url, timeout=15)
body = resp.json()

token = body.get("data", {}).get("token") if "data" in body else None
if token:
    display(HTML(f'<p>&#9989; <strong>SSO login succeeded — Redis fix is working</strong></p>'))
    print(f"Token prefix: {token[:20]}...")
else:
    msg = body.get("message", "")
    display(HTML(f'<p>&#10060; <strong>SSO login failed: {msg}</strong></p>'))
    if "invalid" in msg.lower():
        display(HTML(
            '<p>The PKCE callback likely failed with "Code verifier required."</p>'
            '<p>This confirms the LocMemCache bug. Fix: set Django CACHES default to Redis.</p>'
        ))

## Error 2 (P1): "State is invalid"

The client used `graphistry.register(is_sso_login=True)` with the default
`sso_timeout=50`, which polls for the JWT token immediately. Since the user
hasn't completed login yet, every poll returns "State is invalid."

**To reproduce:** Run Cell 5 — it polls immediately without waiting for login.

In [ ]:
# Cell 5 — Error 2: "State is invalid" (polling race)
#
# Initiates SSO then immediately polls for token.
# Simulates sso_timeout=15 (shortened from default 50).
# Since no one logs in, every poll returns "State is invalid."

import time

POLL_TIMEOUT = 15  # Shortened from default 50 for demo

# Initiate fresh SSO
url = sso_login_url(BASE_URL, ORG_NAME, IDP_NAME)
resp = req.post(url, timeout=15)
body = resp.json()
data = body.get("data", {})
error2_state = data.get("state", "")
error2_auth_url = data.get("auth_url", "")

if not error2_state:
    error_msg = body.get("message") or json.dumps(body)
    display(HTML(f'<p>&#10060; SSO initiation failed: {error_msg}</p>'))
else:
    display(HTML(f'<h3>Error 2: State is invalid</h3>'))
    display(HTML(f'<p>NOT opening auth URL — simulating race condition</p>'))
    display(HTML(f'<p>Polling with sso_timeout={POLL_TIMEOUT}s (client default is 50s)...</p>'))

    poll_url = token_poll_url(BASE_URL, error2_state)
    elapsed = 0
    interval = 2
    last_msg = ""

    while elapsed < POLL_TIMEOUT:
        resp = req.get(poll_url, timeout=15)
        body = resp.json()
        msg = body.get("message", "")
        token = body.get("data", {}).get("token") if "data" in body else None

        if token:
            print(f"[{elapsed}s] Unexpected: got token")
            break

        if msg != last_msg:
            print(f"[{elapsed}s] Waiting for token: {msg}")
            last_msg = msg

        time.sleep(interval)
        elapsed += interval

    if not token:
        display(HTML(f'<p>&#10060; <strong>Timed out after {POLL_TIMEOUT}s — "{last_msg}"</strong></p>'))
        display(HTML(
            '<p>This is what the client saw:</p>'
            '<pre>Exception: State is invalid\n'
            'SsoRetrieveTokenTimeoutException: [SSO] Get token timeout</pre>'
            '<p><strong>Fix:</strong> Use <code>sso_timeout=None</code> (non-blocking mode)</p>'
        ))

## Error 3 (P2): `login()` missing username/password

The client called `graphistry.login()` instead of
`graphistry.register(is_sso_login=True)`. The `login()` method requires
username and password positional arguments.

**To reproduce:** Run Cell 6.

In [ ]:
# Cell 6 — Error 3: login() missing username/password

import graphistry

display(HTML('<h3>Error 3: login() missing username/password</h3>'))
display(HTML('<p>Calling <code>graphistry.login()</code> with no arguments...</p>'))

try:
    graphistry.login()
except TypeError as e:
    display(HTML(f'<p>&#9989; <strong>TypeError raised (as expected):</strong></p>'))
    display(HTML(f'<pre>{e}</pre>'))
    display(HTML(
        '<p>This matches the client error:</p>'
        '<pre>TypeError: GraphistryClient.login() missing 2 required\n'
        'positional arguments: &apos;username&apos; and &apos;password&apos;</pre>'
        '<p><strong>Fix:</strong> Use <code>graphistry.register(is_sso_login=True, ...)</code> instead</p>'
    ))
except Exception as e:
    display(HTML(f'<p>Different error: {type(e).__name__}: {e}</p>'))

## Fixed Flow: Non-blocking SSO

The correct pattern for Databricks:
1. `graphistry.register(is_sso_login=True, sso_timeout=None)` — non-blocking
2. Click the SSO link and complete login in browser
3. `graphistry.sso_get_token()` in a separate cell

This avoids both Error 1 (less server load = less worker switching) and
Error 2 (no premature polling).

In [ ]:
# Cell 7 — Fixed flow: register with non-blocking SSO

import graphistry

display(HTML('<h3>Fixed Flow: Non-blocking SSO</h3>'))

graphistry.register(
    api=3,
    protocol=PROTOCOL,
    server=SERVER,
    is_sso_login=True,
    org_name=ORG_NAME,
    idp_name=IDP_NAME,
    sso_timeout=None,             # Non-blocking
    sso_opt_into_type="display",
)

display(HTML('<p><strong>Click the SSO link above, complete login, then run Cell 8.</strong></p>'))

In [ ]:
# Cell 8 — Get token after login

token = graphistry.sso_get_token()

if token:
    display(HTML(f'<p>&#9989; <strong>SSO login succeeded</strong></p>'))
    print(f"Token prefix: {token[:20]}...")

    is_valid = graphistry.verify_token()
    if is_valid:
        display(HTML('<p>&#9989; Token verified with server</p>'))
    else:
        display(HTML('<p>&#10060; Token verification failed</p>'))
else:
    display(HTML('<p>&#10060; <strong>No token.</strong> Complete SSO login first, then re-run this cell.</p>'))

In [ ]:
# Cell 9 — Summary

html = '<h3>Error Reproduction Summary</h3>'
html += '<table style="border-collapse:collapse;">'
html += '<tr style="background:#eee;"><th style="padding:4px 8px;">Error</th><th style="padding:4px 8px;">Description</th><th style="padding:4px 8px;">Reproduced?</th><th style="padding:4px 8px;">Fix</th></tr>'

rows = [
    ('P0', 'Code verifier required', 'Run Cells 3-4', 'Django CACHES default → Redis'),
    ('P1', 'State is invalid', 'Cell 5 shows timeout', 'sso_timeout=None'),
    ('P2', 'login() TypeError', 'Cell 6 shows TypeError', 'Use register(is_sso_login=True)'),
]

for err, desc, repro, fix in rows:
    html += f'<tr><td style="padding:4px 8px;"><strong>{err}</strong></td>'
    html += f'<td style="padding:4px 8px;">{desc}</td>'
    html += f'<td style="padding:4px 8px;">{repro}</td>'
    html += f'<td style="padding:4px 8px;"><code>{fix}</code></td></tr>'

html += '</table>'

try:
    tok = graphistry.api_token()
    if tok:
        html += '<p style="color:green;">&#9989; <strong>Fixed flow working — token acquired</strong></p>'
    else:
        html += '<p style="color:orange;">&#9888; Fixed flow not tested yet (run Cells 7-8)</p>'
except Exception:
    html += '<p style="color:orange;">&#9888; Fixed flow not tested yet (run Cells 7-8)</p>'

display(HTML(html))